In [15]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# ==========================================
# 1. LOAD DATASETS
# ==========================================
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Create copies for feature engineering
X_train = original_df.drop(columns=["Session_ID", "Revenue"]).copy()
y_train = original_df["Revenue"].astype(int)

X_test = test_df.drop(columns=["Session_ID"]).copy()

# ==========================================
# 2. FEATURE ENGINEERING (Domain Interactions)
# ==========================================
for df in [X_train, X_test]:
    df['Duration_Per_Product'] = df['ProductRelated_Duration'] / (df['ProductRelated'] + 1)
    df['Admin_Duration_Per_Page'] = df['Administrative_Duration'] / (df['Administrative'] + 1)
    df['PageValue_x_Exit'] = df['PageValues'] * df['ExitRates']

# ==========================================
# 3. FEATURE SEPARATION & TYPE CONVERSION
# ==========================================
# Include discrete numerical IDs into categorical list for One-Hot Encoding
cat_cols = ['Month', 'VisitorType', 'Weekend', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']
num_cols = [col for col in X_train.columns if col not in cat_cols]

for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

# ==========================================
# 4. PREPROCESSING PIPELINE
# ==========================================
# Yeo-Johnson power transformation normalizes skewed durations and page values
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power_transform', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Build processed_df for checkpoint validation
X_train_proc = preprocessor.fit_transform(X_train)
feature_names = preprocessor.get_feature_names_out()

processed_df = pd.DataFrame(X_train_proc, columns=feature_names)
processed_df['Revenue'] = y_train.values

# ==========================================
# 5. TRAIN MODEL WITH FINE-TUNED REGULARIZATION
# ==========================================
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=5000, random_state=42))
])

param_grid = {
    'classifier__C': [0.5, 1.0, 2.0, 5.0],
    'classifier__solver': ['lbfgs'],
    'classifier__class_weight': [None]
}

grid_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Assign checkpoint variables
model = grid_search
predictions = model.predict(X_test)

# Print accuracy improvements
train_acc = accuracy_score(y_train, model.predict(X_train))
val_acc = model.best_score_

print("=" * 45)
print(f"New Training Set Accuracy:           {train_acc * 100:.2f}%")
print(f"New 5-Fold CV (Validation) Accuracy: {val_acc * 100:.2f}%")
print("=" * 45)

# ==========================================
# 6. GENERATE SUBMISSION FILE
# ==========================================
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat([checkpoints, prediction_output], ignore_index=True)
submission.to_csv("submission2.csv", index=False)

print("submission2.csv updated successfully!")
print(checkpoints)

New Training Set Accuracy:           88.64%
New 5-Fold CV (Validation) Accuracy: 88.33%
submission2.csv updated successfully!
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  79
6  row_retained_percent               100.0
7            model_name  LogisticRegression
